# Top 100 Countries by Population

This notebook fetches country population data from the REST Countries API and returns the 100 most populous countries.

In [3]:
import pandas as pd
import requests

In [ ]:
API_URL = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL"

params = {
    "format": "json",
    "per_page": 20000,
    "date": "2024"
}

# Build a whitelist of actual countries (exclude aggregate groups).
country_meta_url = "https://api.worldbank.org/v2/country"
country_meta = requests.get(
    country_meta_url,
    params={"format": "json", "per_page": 400},
    timeout=60,
)
country_meta.raise_for_status()
country_payload = country_meta.json()

if not isinstance(country_payload, list) or len(country_payload) < 2:
    raise RuntimeError(f"Unexpected country metadata shape: {country_payload}")

valid_iso3 = {
    c.get("id")
    for c in country_payload[1]
    if (c.get("region") or {}).get("value") != "Aggregates" and c.get("id")
}

response = requests.get(API_URL, params=params, timeout=60)
response.raise_for_status()
payload = response.json()

if not isinstance(payload, list) or len(payload) < 2:
    raise RuntimeError(f"Unexpected World Bank API response shape: {payload}")

records = payload[1]
rows = []
for rec in records:
    country = (rec.get("country") or {}).get("value")
    country_id = (rec.get("country") or {}).get("id")
    country_iso3 = rec.get("countryiso3code")
    population = rec.get("value")

    if country and population is not None and country_iso3 in valid_iso3:
        rows.append({
            "country": country,
            "code": country_id or "",
            "region": "",
            "population": population
        })

df = pd.DataFrame(rows)
df["population"] = pd.to_numeric(df["population"], errors="coerce").fillna(0).astype("int64")


In [10]:
top_100 = (
    df.sort_values("population", ascending=False)
      .head(100)
      .reset_index(drop=True)
)

top_100.index = top_100.index + 1
top_100.index.name = "rank"

try:
    display(top_100)
except NameError:
    print(top_100.to_string())

,country,code,region,population
rank,,,,
1,India,IN,,1450935791
2,China,CN,,1408975000
3,United States,US,,340003797
4,Indonesia,ID,,283487931
5,Pakistan,PK,,251269164
...,...,...,...,...
96,Hungary,HU,,9562065
97,Austria,AT,,9177982
98,Belarus,BY,,9132629


In [12]:
import os

file_name = "top_100_countries_by_population.csv"
export_df = top_100.reset_index()  # keep rank as a CSV column
saved_paths = []

# Databricks-persistent location (if DBFS is mounted)
dbfs_dir = "/dbfs/FileStore/exports"
if os.path.isdir("/dbfs"):
    os.makedirs(dbfs_dir, exist_ok=True)
    dbfs_path = os.path.join(dbfs_dir, file_name)
    export_df.to_csv(dbfs_path, index=False)
    saved_paths.append(dbfs_path)

# Local fallback/copy for non-Databricks local execution
local_dir = os.path.abspath("./outputs")
os.makedirs(local_dir, exist_ok=True)
local_path = os.path.join(local_dir, file_name)
export_df.to_csv(local_path, index=False)
saved_paths.append(local_path)

print("Saved CSV to:")
for path in saved_paths:
    print(f" - {path}")


Saved CSV to:
 - /workspaces/cs-databricks/notebooks/outputs/top_100_countries_by_population.csv
